In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import Window
import snowflake.snowpark.functions as F

session = get_active_session()

In [ ]:
df_contract = session.table("BIGDATA_DB.RAW.CONTRACTS")
df_map_user = session.table("BIGDATA_DB.STAGING.MAP_USER")
df_transfers_indexed = session.table("BIGDATA_DB.STAGING.TRANSFERS_INDEXED")


In [ ]:
from snowflake.snowpark import functions as F

# Tao 2 luong token ra va vao
df_inflow = df_transfers_indexed.select(
    F.lower(F.col("to_user_id")).alias("user_id"),
    F.lower(F.col("token_id")).alias("token_id"),
    F.col("adjust_value").cast("double").alias("value"),
    F.col("block_timestamp")
)

df_outflow = df_transfers_indexed.select(
    F.lower(F.col("from_user_id")).alias("user_id"),
    F.lower(F.col("token_id")).alias("token_id"),
    (F.col("adjust_value").cast("double") * F.lit(-1.0)).alias("value"),
    F.col("block_timestamp")
)

df_inflow.limit(5).show()
df_outflow.limit(5).show()


In [ ]:
df_transfer_flow = df_inflow.union(df_outflow)
df_portfolios = (
    df_transfer_flow
    .group_by("user_id", "token_id")
    .agg(
        F.sum("value").alias("balance"),
        F.count("*").alias("tx_count"),
        F.max("block_timestamp").alias("last_active")
    )
    .filter(F.col("balance") > 0)
    .filter(F.col("user_id").is_not_null())
)

df_portfolios_sample = df_portfolios.limit(20)
df_portfolios_sample.show()


In [ ]:
df_portfolios.write.mode("overwrite").save_as_table("BIGDATA_DB.STAGING.USER_PORTFOLIOS")

In [ ]:
df_portfolios = session.table("BIGDATA_DB.STAGING.USER_PORTFOLIOS")

In [ ]:
df_portfolios_join = (
    df_portfolios
    .join(
        df_map_user,
        on=(df_portfolios["user_id"] == df_map_user["user_id"])
    )
    .select(
        df_portfolios["user_id"].alias("user_id"),
        df_map_user["user_address"].alias("user_address"),
        df_portfolios["token_id"].alias("token_id"),
        df_portfolios["BALANCE"].alias("BALANCE"),
        df_portfolios["TX_COUNT"].alias("TX_COUNT"),
        df_portfolios["LAST_ACTIVE"].alias("LAST_ACTIVE")
    )
)

df_remove_contract = (
    df_portfolios_join
    .join(
        df_contract,
        on=(df_portfolios_join["user_address"] == df_contract["address"]),
        how="leftanti"
    )
    .select(
        df_portfolios_join["user_id"].alias("user_id"),
        df_portfolios_join["token_id"].alias("token_id"),
        df_portfolios_join["BALANCE"].alias("BALANCE"),
        df_portfolios_join["TX_COUNT"].alias("TX_COUNT"),
        df_portfolios_join["LAST_ACTIVE"].alias("LAST_ACTIVE")
    )
)

df_remove_contract.show(30)

In [ ]:
df_remove_contract.count()

In [ ]:
df_user_total_tx = (
    df_remove_contract
    .group_by("user_id")
    .agg(F.sum("TX_COUNT").alias("total_tx_count"))
)

df_remove_contract = (
    df_remove_contract
    .join(
        df_user_total_tx.filter(F.col("total_tx_count") >= 10000),
        on="user_id",
        how="leftanti"
    )
)

df_remove_contract.show(30)

In [ ]:
df_user_token_count = (
    df_remove_contract
    .group_by("user_id")
    .agg(
        F.count_distinct("token_id").alias("token_count"),
        F.sum("tx_count").alias("total_tx_count"),
    )
    .order_by(F.desc("token_count"), F.desc("TOTAL_TX_COUNT"))
)

In [ ]:
df_user_token_count.limit(20).show()
print("So user:", df_user_token_count.count())

In [ ]:
df_pagerank_score = session.table("BIGDATA_DB.STAGING.PAGERANK_FINAL_RESULTS")

df_user_token_count_pagerank = (
    df_user_token_count
    .join(
        df_pagerank_score,
        on=(df_user_token_count["USER_ID"] == df_pagerank_score["ID"]),
        how="inner"
    )
    .select(
        df_user_token_count["USER_ID"].alias("USER_ID"),
        df_user_token_count["TOKEN_COUNT"].alias("TOKEN_COUNT"),
        df_user_token_count["TOTAL_TX_COUNT"].alias("TOTAL_TX_COUNT"),
        df_pagerank_score["PAGERANK"].alias("PAGERANK_SCORE")
    )
    .sort(F.col("PAGERANK").desc())
)

df_user_token_count_pagerank.show()

In [ ]:
total_users = df_user_token_count_pagerank.count()
top_n = max(1, int(total_users * 0.05))

w_pr = Window.order_by(F.col("PAGERANK_SCORE").desc())

df_top_5_percent_pagerank = (
    df_user_token_count_pagerank
    .with_column("rank_pr", F.row_number().over(w_pr))
    .filter(F.col("rank_pr") <= F.lit(top_n))
    .drop("rank_pr")
)



In [ ]:
df_remove_contract_5_percent.write.mode("overwrite").save_as_table("BIGDATA_DB.STAGING.USER_PORTFOLIOS)